# Desafio Agentes IA — Roteiro + Thumbnails para vídeos de games (CrewAI)

Três agentes trabalhando em sequência:

1. **Roteirista de Vídeo**: pesquisa na web e escreve o roteiro completo.
2. **Criador de Thumbnail**: lê o roteiro, gera **3 thumbnails** com DALL·E e escolhe a melhor.
3. **Revisor**: revisa o texto do roteiro e monta o documento final (roteiro + thumbnails).

**Chaves necessárias** (o notebook pede via `getpass`):
- `OPENAI_API_KEY`: LLM e geração de imagens
- `SERPER_API_KEY`: pesquisa na web (plano grátis em serper.dev)

## 1. Instalação

Se o Colab reclamar de dependências, reinicie o ambiente de execução e rode de novo a partir da célula 2.

In [ ]:
!pip install -q crewai crewai-tools

## 2. Chaves de API

In [ ]:
import os, getpass

for key in ["OPENAI_API_KEY", "SERPER_API_KEY"]:
    if key not in os.environ:
        os.environ[key] = getpass.getpass(f"{key}: ")

## 3. Configuração, LLM e ferramentas

In [ ]:
from crewai import Agent, Task, Crew, Process, LLM
from crewai_tools import SerperDevTool, DallETool

llm = LLM(model="gpt-4o-mini", temperature=0.7)

search_tool = SerperDevTool()

# DALL-E 3 gera 1 imagem por chamada; o agente chama a ferramenta 3 vezes.
# Se o modelo "dall-e-3" não estiver disponível na sua conta, troque o valor abaixo.
IMAGE_MODEL = "dall-e-3"
dalle_tool = DallETool(model=IMAGE_MODEL, size="1792x1024", quality="standard", n=1)

## 4. Agentes

In [ ]:
roteirista = Agent(
    role="Roteirista de Vídeo",
    goal="Pesquisar e escrever um roteiro completo e envolvente para um vídeo de YouTube sobre: {query}",
    backstory=(
        "Você é um especialista em criação de conteúdo e storytelling para canais de videogames no YouTube. "
        "Você pesquisa dados atuais e confiáveis, estrutura o vídeo com gancho forte, desenvolvimento "
        "dinâmico e chamada para ação, e escreve em português do Brasil, com linguagem natural e animada."
    ),
    tools=[search_tool],
    llm=llm,
    verbose=True,
)

criador_thumbnail = Agent(
    role="Criador de Thumbnail",
    goal="Criar 3 opções de thumbnail chamativas, inspiradas no roteiro, e escolher a melhor",
    backstory=(
        "Você é um designer gráfico com anos de experiência em thumbnails de YouTube para o nicho de games. "
        "Sabe que thumbnails boas têm poucos elementos, cores de alto contraste, rostos/expressões fortes, "
        "personagens ou jogos reconhecíveis e, no máximo, 3 a 4 palavras de texto."
    ),
    tools=[dalle_tool],
    llm=llm,
    verbose=True,
)

revisor = Agent(
    role="Revisor",
    goal="Revisar o roteiro e entregar o documento final com roteiro revisado e thumbnails",
    backstory=(
        "Você é um editor de conteúdo rigoroso: corrige erros de português, melhora clareza e ritmo, "
        "corta repetições e garante que o texto soe natural para ser falado em vídeo."
    ),
    llm=llm,
    verbose=True,
)

## 5. Tarefas

In [ ]:
tarefa_roteiro = Task(
    description=(
        "Pesquise na web sobre o tema '{query}' e escreva o roteiro completo de um vídeo de YouTube. "
        "O roteiro deve conter: título sugerido, gancho (primeiros 15 segundos), introdução, "
        "desenvolvimento dividido em seções/tópicos com o que falar em cada um, conclusão e chamada para ação. "
        "Use informações verificadas na pesquisa e indique o tempo aproximado de cada seção."
    ),
    expected_output="Roteiro completo em português do Brasil, em Markdown, pronto para ser gravado.",
    agent=roteirista,
)

tarefa_thumbnail = Task(
    description=(
        "Com base no roteiro recebido, crie 3 thumbnails DIFERENTES entre si (conceitos, composição e cores distintos). "
        "Para cada uma, chame a ferramenta de geração de imagem UMA vez com um prompt detalhado em inglês "
        "(cena, jogo/personagem, expressão, cores, iluminação, estilo, espaço para texto curto). "
        "Depois escolha a melhor e justifique em 2 ou 3 frases (clareza, contraste, curiosidade)."
    ),
    expected_output=(
        "Para cada thumbnail (1, 2 e 3): descrição do conceito, o prompt usado e a URL da imagem gerada. "
        "No final: qual foi a escolhida e por quê."
    ),
    agent=criador_thumbnail,
    context=[tarefa_roteiro],
)

tarefa_revisao = Task(
    description=(
        "Revise o roteiro (ortografia, gramática, clareza, ritmo, coerência). Depois escreva o documento final em Markdown com: "
        "1) o título do vídeo; 2) o roteiro revisado completo; 3) a seção 'Thumbnails' com as 3 opções "
        "(descrição + URL da imagem, no formato ![Thumbnail N](URL)) e destaque qual foi a escolhida e o motivo."
    ),
    expected_output="Documento final em Markdown contendo roteiro revisado e as 3 thumbnails, com a escolhida destacada.",
    agent=revisor,
    context=[tarefa_roteiro, tarefa_thumbnail],
    output_file="relatorio_final.md",
)

## 6. Crew

In [ ]:
crew = Crew(
    agents=[roteirista, criador_thumbnail, revisor],
    tasks=[tarefa_roteiro, tarefa_thumbnail, tarefa_revisao],
    process=Process.sequential,
    verbose=True,
)

## 7. Execução

Troque o tema à vontade.

In [ ]:
resultado = crew.kickoff(inputs={"query": "Melhores jogos de 2020"})

## 8. Resultado final

In [ ]:
from IPython.display import Markdown, display

display(Markdown(resultado.raw))

## 9. Thumbnails geradas

Extrai as URLs das imagens da saída do Criador de Thumbnail e mostra as três.

In [ ]:
import re
from IPython.display import Image, display

texto = tarefa_thumbnail.output.raw
urls = list(dict.fromkeys(re.findall(r"https?://[^\s)\]\">]+", texto)))

for i, url in enumerate(urls, 1):
    print(f"Thumbnail {i}: {url}")
    display(Image(url=url, width=640))

> As URLs do DALL·E expiram em cerca de 1 hora. Se quiser guardar as imagens, baixe-as logo após a execução.

## Observações

Preencha após rodar:
- O roteiro usou informações pesquisadas de fato (veja o log do Roteirista)?
- As 3 thumbnails ficaram realmente diferentes? A escolhida faz sentido?
- O que o Revisor mudou em relação ao roteiro original?